# M01 Toy Models of Superposition


## Toy setup

Compress four sparse concepts into a two-dimensional hidden space and inspect how overlap appears.


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(7)

num_features = 4
hidden_dim = 2
encoder = torch.nn.Linear(num_features, hidden_dim, bias=False)
decoder = torch.nn.Linear(hidden_dim, num_features, bias=False)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.05)

for step in range(500):
    active = (torch.rand(512, num_features) < 0.22).float()
    strengths = torch.rand(512, num_features)
    batch = active * strengths
    hidden = encoder(batch)
    recon = decoder(hidden)
    loss = torch.nn.functional.mse_loss(recon, batch) + 0.002 * hidden.abs().mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

samples = torch.eye(num_features)
hidden_samples = encoder(samples).detach()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(hidden_samples[:, 0], hidden_samples[:, 1], s=130, c=range(num_features), cmap="tab10")
for index, point in enumerate(hidden_samples):
    axes[0].annotate(f"f{index}", (point[0].item(), point[1].item()))
axes[0].set_title("Feature positions in 2D hidden space")
axes[0].axhline(0, color="0.8", linewidth=1)
axes[0].axvline(0, color="0.8", linewidth=1)

axes[1].imshow(decoder.weight.detach().T, cmap="coolwarm", aspect="auto")
axes[1].set_title("Decoder weights")
axes[1].set_xlabel("Hidden dimension")
axes[1].set_ylabel("Original feature")
plt.tight_layout()

print("Final loss:", float(loss.detach()))
print("Hidden representations:\n", hidden_samples)


## Takeaway

Once the hidden space is smaller than the concept set, overlap stops being a bug and becomes the expected geometry.


## Turn this notebook into research output

Research worksheet means this notebook is not complete when the cells finish. Use the templates in /templates and leave behind written outputs.


### Before you run

- Choose one variable family to change: hidden dimension, sparsity, or loss weight.
- Predict how the geometry should change before you run the sweep.
- Write the baseline settings into the experiment-log template.


### After you run

- State the geometric reason overlap appears in your run.
- Separate what the plot shows directly from the story you tell about neurons.


### Ship these artifacts

- One experiment log with baseline and one sweep.
- One annotated plot of the hidden geometry.
- One 100-200 word memo on why neuron semantics break.


## Self-check questions

- Why is superposition better understood as a geometric outcome rather than merely a training bug?
- In your toy-model sweep, which change most clearly altered concept overlap: hidden dimension, sparsity, or the loss term?
- If someone insists on treating a single neuron as a stable semantic unit, which piece of evidence from this article would you use to push back?
- If you cannot answer at least two of these without rereading the note, revisit the article and your written outputs.
